<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 08: Proyecto Capstone</p>
  </div>
</div>

# Proyecto Final · Plantilla CRISP-DM

Esta es la **plantilla** de tu proyecto final. Clónala y complétala con **tu** caso:

1. Elige **un** problema: **regresión**, **clasificación** o **forecasting** (serie de tiempo).
2. Elige un **dataset del menú** (o datos propios, **sin datos personales/confidenciales**).
3. Recorre las **fases de CRISP-DM** de abajo, reemplazando los ejemplos y los `# TODO` por tu caso.

**Reglas** (ver la pauta del Proyecto Final): fija la **semilla** (`SESSION_ID`), carga los datos por **URL estable**, haz **todo el modelado con PyCaret / StatsForecast**, y **declara el uso de IA** en el anexo. El notebook debe correr completo con *Ejecutar todo* y dar **los mismos resultados**.

## Preparación del entorno

Instala las librerías del módulo. Usarás **PyCaret** (tabular) **o** **StatsForecast** (series), según tu problema.

In [ ]:
%%capture
!pip install -q pycaret==4.0.0a8 "statsforecast==2.0.3" "utilsforecast==0.2.15" \
    fg-data-profiling missingno plotly ipywidgets shap "setuptools<81"
# Opcional (más modelos para compare_models): !pip install -q xgboost catboost lightgbm optuna

## Setup técnico

Importaciones y configuración general (no es contenido del proyecto).

In [ ]:
%matplotlib inline
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px

SESSION_ID = 42   # fija la semilla: tu trabajo debe ser REPLICABLE

## Elige tu problema y carga tus datos

- **Tipo de problema**: regresión / clasificación / forecasting.
- **Dataset**: del menú de la pauta (o propio, sin datos sensibles).

**Las celdas de carga de cada dataset del menú están en el notebook `datasets`** (una por dataset, con las columnas ya en español y los tipos resueltos). Ábrelo, **copia la celda del tuyo** y pégala en la celda de abajo:

[![Abrir el notebook de datasets en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastiancontz/ust-fundamentos-machine-learning-colab/blob/main/ediciones/2026/notebooks/datasets.ipynb)

Si usas **datos propios**, cárgalos tú (ajusta a tu archivo):

```python
URL = "https://.../tu_dataset.csv"
df = pd.read_csv(URL)     # ajusta sep / encoding si hace falta
df.head()
```

In [ ]:
# Pega aquí la celda de carga de tu dataset (desde el notebook `datasets`), o carga tus datos propios.


## Fase 1 · Comprender el negocio

Antes de tocar un modelo, define (en texto):

- **Pregunta de negocio**: ¿qué **decisión** ayuda a tomar tu modelo?
- **Métrica ligada a la decisión**: la que responde *tu* pregunta (no la que "se ve mejor"). Distingue el éxito **técnico** del éxito de **negocio**.
- **Baseline**: la vara mínima a superar — mayoritario/promedio (tabular) o Naive/SeasonalNaive (series). *Un modelo que no le gana al baseline no aporta.*

> _Escribe aquí tu pregunta de negocio, tu métrica y tu baseline._

## Fase 2 · Comprender los datos (EDA y *outliers*)

Explora antes de modelar: tipos de variables, distribuciones, correlaciones, calidad. Detecta y **decide** qué hacer con faltantes y *outliers* (no borres por reflejo; explica tu decisión).

```python
df.info(); df.describe()

# reporte automático de EDA (como en la Clase 2)
from data_profiling import ProfileReport
ProfileReport(df).to_notebook_iframe()

# outliers a ojo (boxplot) y distribuciones (interactivo)
px.box(df, y="tu_variable_numerica")
px.histogram(df, x="tu_variable")
```

In [ ]:
# TODO: tu EDA + análisis de outliers


## Fase 3 · Preparar los datos (sin fuga)

**Cuida la fuga de datos**: el preprocesamiento no debe "ver" el test, y ninguna variable debe "saber la respuesta".

**Si tu problema es TABULAR** — deja que PyCaret prepare (train/test, imputación, escalado):

```python
from pycaret.tasks import ClassificationExperiment   # o RegressionExperiment
exp = ClassificationExperiment(
    target="tu_target", session_id=SESSION_ID, train_size=0.7,
    fold=5, normalize=True, fold_strategy="stratifiedkfold",   # estratificada en clasificación
).fit(df)
```

**Si tu problema es FORECASTING** — deja la serie en formato `unique_id` / `ds` / `y`:

```python
serie = df.rename(columns={"tu_fecha": "ds", "tu_valor": "y"})
serie["unique_id"] = "serie_1"
serie = serie[["unique_id", "ds", "y"]].sort_values("ds")
```

In [ ]:
# TODO: prepara tus datos (setup de PyCaret o formato unique_id/ds/y)


## Fase 4 · Modelar y comparar contra el *baseline*

Compara varios modelos, elige por las **métricas** (no por corazonada) y afina donde aplique.

**TABULAR (PyCaret):**

```python
mejores = exp.compare_models(sort="AUC")          # regresión: sort="RMSE"
base = exp.create_model("rf")                      # crea el modelo que elijas del ranking; vive en base.pipeline
afinado = exp.tune_model(base.pipeline, n_iter=30, optimize="AUC")   # luego compara base vs afinado en el holdout y usa el mejor
```

**FORECASTING (StatsForecast):**

```python
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA
sf = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=12), AutoETS(season_length=12), AutoARIMA(season_length=12)],
    freq="MS", n_jobs=-1)     # ajusta season_length y freq a tu serie
```

In [ ]:
# TODO: entrena y compara tus modelos contra el baseline


## Fase 4b · Segmentación con clustering — **solo TABULAR**

Suma una capa de segmentación (como en la Clase 5). **En forecasting este paso se OMITE**: se reemplaza por leer la anatomía de la serie (tendencia, estacionalidad, quiebres).

```python
from pycaret.tasks import ClusteringExperiment
cexp = ClusteringExperiment(session_id=SESSION_ID, normalize=True).fit(df)
km = cexp.create_model("kmeans", n_clusters=4)     # OJO: es n_clusters (num_clusters se ignora en silencio)
etiquetado = cexp.assign_model(km.pipeline)         # perfila los grupos y ponles un nombre accionable
```

In [ ]:
# TODO (solo tabular): clustering y perfilado de segmentos


## Fase 5 · Evaluar e interpretar

Evaluación **honesta** (holdout intacto o validación temporal) y luego **interpreta** para conectar con la decisión.

**TABULAR:**

```python
exp.predict_model(base.pipeline).metrics             # métricas en el holdout (una sola vez)
exp.plot_model(base.pipeline, plot="shap_beeswarm")  # qué variables mueven la aguja
exp.plot_model(base.pipeline, plot="pdp", feature="tu_variable")   # gráfico de dependencia
```

> **Ojo (como en la Clase 6):** para que `shap_beeswarm` y `pdp` funcionen, el modelo necesita **features numéricas** y el **target en 0/1** — mapea el target a 0/1 y deja las categóricas ya codificadas (en `float`). Si no, dan un error de tipos.

**FORECASTING (validación temporal + intervalos):**

```python
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, mape
H = 12
cv = sf.cross_validation(df=serie, h=H, n_windows=3, step_size=12)
evaluate(cv.drop(columns="cutoff"), metrics=[mae, mape],
         models=["Naive", "SeasonalNaive", "AutoETS", "AutoARIMA"])
fc = sf.forecast(df=serie, h=H, level=[80, 95])       # pronóstico + intervalos
```

In [ ]:
# TODO: evalúa (sin fuga) e interpreta


## Fase 5b · Del modelo al valor (costos y beneficios)

Traduce la métrica a **pesos**: pon un costo/beneficio a cada acierto y error y compáralo con "no hacer nada". La métrica informa; el **costo decide**.

> _Escribe aquí tu cuenta de valor (aunque sea una tabla simple de costos/beneficios)._

In [ ]:
# TODO: tu marco de valor esperado


## Fase 6 · Comunicar y guardar

**Guarda** el modelo final para usarlo con datos nuevos (tabular):

```python
final = exp.finalize_model(base.pipeline)
exp.save_model(final.pipeline, "mi_modelo")   # luego se recupera con load_model
```

**Comunica** la decisión en el **informe** (≤ 5 páginas, audiencia no técnica): problema → EDA → comparación de modelos → valor → recomendación. **Gráficos antes que viñetas.**

In [ ]:
# TODO (tabular): finalize_model + save_model


## Anexo · Declaración de uso de IA

El uso de IA está permitido, pero **debe declararse** (si no, descuenta). Completa la tabla con **cada uso**:

| ¿Dónde / para qué? | Modelo | Prompt (resumen) |
|---|---|---|
|  |  |  |

## Checklist de entrega

- [ ] Corre completo con *Ejecutar todo*, con semillas fijas y datos por URL.
- [ ] Modelado hecho con PyCaret / StatsForecast (no otra librería sin justificar).
- [ ] EDA + *outliers* · baseline comparado · métricas presentadas según el tipo.
- [ ] Mejor modelo afinado (donde aplique).
- [ ] Clustering (tabular) o lectura de la serie (forecasting).
- [ ] Interpretabilidad (SHAP/dependencia o componentes/intervalos) conectada a una decisión.
- [ ] Marco de valor (costos/beneficios).
- [ ] Modelo guardado (tabular).
- [ ] Informe ≤ 5 páginas + anexo de uso de IA.

## Atribución de datos

**Creador:** CNE, MINSAL-DEIS, Department for Transport (GB), datos.gob.cl y Department of Transport and Planning (Victoria).
**Fuente:** [catálogo de datos abiertos de Chile](https://datos.gob.cl/), [data.gov.uk](https://www.data.gov.uk/) y [datos abiertos de Victoria](https://discover.data.vic.gov.au/).
**Licencia:** combinación de [CC0](https://creativecommons.org/publicdomain/zero/1.0/), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), [OGL v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/) y [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/), según el dataset elegido.
**Modificación:** columnas traducidas, tipos corregidos, recorte de campos y agregaciones temporales documentadas en el catálogo del proyecto.